In [0]:
# Imports

# Imports
from pyspark.sql.functions import col, trim, lower, current_timestamp, lit, expr, split, regexp_replace
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
 

In [0]:
# Ler a Bronze V2
df_bronze = spark.table("workspace.drs_bronze.gastos_insumos_por_cultura")
display(df_bronze)

In [0]:
# Padronizando os nomes das colunas
df_insumos = df_bronze \
    .withColumnRenamed("categoria", "category") \
    .withColumnRenamed("insumo",    "input_name") \
    .withColumnRenamed("unidade",   "unit")
 
print(f"Total registros Bronze (wide): {df_insumos.count()}")
display(df_insumos)

In [0]:
from pyspark.sql.functions import col, trim, lower, current_timestamp, lit, expr, split, regexp_replace, when

# UNPIVOT — transforma colunas de cultura em linhas
# De: 1 linha por insumo com colunas soja / milho / sorgo (valores "60-80")
# Para: 1 linha por insumo x cultura (45 linhas no total)
 
df_unpivot = df_insumos.select(
    col("category"),
    col("input_name"),
    col("unit"),
    col("ingestion_timestamp"),
    col("source_file"),
    expr("stack(3, 'soja', soja, 'milho', milho, 'sorgo', sorgo) AS (culture, dose_range)")
)
 
# Separar "60-80" em dose_min e dose_max
df_unpivot = (
    df_unpivot
    .withColumn("dose_range", regexp_replace(col("dose_range"), r"[–—]", "-"))
    .withColumn("dose_range", regexp_replace(col("dose_range"), r",", "."))
    # FIX: try_cast via expr — tolera vazios e malformados → vira NULL
    .withColumn("dose_min",
        when(col("dose_range").contains("-"),
             expr("try_cast(split(dose_range, '-')[0] AS double)"))
        .otherwise(expr("try_cast(dose_range AS double)"))
    )
    .withColumn("dose_max",
        when(col("dose_range").contains("-"),
             expr("try_cast(split(dose_range, '-')[1] AS double)"))
        .otherwise(expr("try_cast(dose_range AS double)"))
    )
    .drop("dose_range")
)
 
print(f"Total registros após unpivot (long): {df_unpivot.count()}")
display(df_unpivot)

In [0]:
# Transformar/Converter colunas para o padrão
from pyspark.sql.functions import initcap

df_silver = (
    df_unpivot
    .withColumn("category",   lower(trim(col("category"))))
    .withColumn("input_name", lower(trim(col("input_name"))))
    .withColumn(
        "culture",
        initcap(trim(col("culture")))   # ← Milho / Soja / Sorgo
    )
    .withColumn("unit",       lower(trim(col("unit"))))
    .withColumn("dose_min",   col("dose_min").cast("double"))
    .withColumn("dose_max",   col("dose_max").cast("double"))
)

In [0]:
# Criando o campo source_file
df_silver = df_silver.withColumn("source_file", lit("gastos_insumos_por_cultura.csv"))
 
# COMMAND ----------
 
# Separando registros inválidos para quarentena
df_invalid = df_silver.filter("""
    culture    IS NULL
    OR input_name IS NULL
    OR category   IS NULL
    OR (dose_min IS NULL AND dose_max IS NULL)
""")
 
print(f"Total registros inválidos: {df_invalid.count()}")

In [0]:
# Salvando quarentena da Silver
df_invalid.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("workspace.drs_silver.gastos_insumos_por_cultura_quarantine")

In [0]:
# Criando o campo ingestion_timestamp
df_silver = df_silver.withColumn("ingestion_timestamp", current_timestamp())

In [0]:
# Filtrando valores válidos
df_silver_filtered = (
    df_silver
    .filter(
        col("culture").isNotNull() &
        col("input_name").isNotNull() &
        col("category").isNotNull()
    )
)
 
print(f"Total registros válidos: {df_silver_filtered.count()}")
 

In [0]:
# Criando campos de auditoria
df_insumos = df_silver_filtered \
    .withColumn("silver_processed_timestamp", current_timestamp()) \
    .withColumn("created_at",                 current_timestamp()) \
    .withColumn("updated_at",                 current_timestamp())

In [0]:
# Deduplicação por chave de negócio
# Chave: category + input_name + culture (alinhada com o MERGE abaixo)
 
window_spec = Window.partitionBy(
    "category",
    "input_name",
    "culture"
).orderBy(
    col("ingestion_timestamp").desc()
)
 
df_insumos = (
    df_insumos
    .withColumn("row_num", row_number().over(window_spec))
    .filter(col("row_num") == 1)
    .drop("row_num")
)
 
print(f"Total após deduplicação: {df_insumos.count()}")
 

In [0]:
# Validando schema
df_insumos.printSchema()

In [0]:
# Salvando staging table
df_insumos.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.drs_silver.gastos_insumos_por_cultura_staging")

In [0]:
# Criando tabela final se não existir
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.drs_silver.gastos_insumos_por_cultura
AS
SELECT * FROM workspace.drs_silver.gastos_insumos_por_cultura_staging WHERE 1 = 0
""")

In [0]:
# MERGE incremental
# Chave: category + input_name + culture (mesma do Window acima)
 
spark.sql("""
MERGE INTO workspace.drs_silver.gastos_insumos_por_cultura AS target
USING workspace.drs_silver.gastos_insumos_por_cultura_staging AS source
 
ON  target.category   = source.category
AND target.input_name = source.input_name
AND target.culture    = source.culture
 
WHEN MATCHED THEN
UPDATE SET
    target.dose_min                   = source.dose_min,
    target.dose_max                   = source.dose_max,
    target.unit                       = source.unit,
    target.ingestion_timestamp        = source.ingestion_timestamp,
    target.source_file                = source.source_file,
    target.silver_processed_timestamp = source.silver_processed_timestamp,
    target.updated_at                 = current_timestamp()
 
WHEN NOT MATCHED THEN
INSERT (
    category,
    input_name,
    culture,
    dose_min,
    dose_max,
    unit,
    ingestion_timestamp,
    source_file,
    silver_processed_timestamp,
    created_at,
    updated_at
)
VALUES (
    source.category,
    source.input_name,
    source.culture,
    source.dose_min,
    source.dose_max,
    source.unit,
    source.ingestion_timestamp,
    source.source_file,
    source.silver_processed_timestamp,
    current_timestamp(),
    current_timestamp()
)
""")

In [0]:
# Validando a tabela Silver final
spark.sql("""
SELECT *
FROM workspace.drs_silver.gastos_insumos_por_cultura
ORDER BY category, input_name, culture
""").display()
 
# COMMAND ----------
 
# Verificar duplicidade pela chave do merge
spark.sql("""
SELECT
    category,
    input_name,
    culture,
    COUNT(*) AS qtd
FROM workspace.drs_silver.gastos_insumos_por_cultura
GROUP BY category, input_name, culture
HAVING COUNT(*) > 1
ORDER BY qtd DESC
""").display()

In [0]:
# Validação quarentena
spark.sql("""
SELECT COUNT(*) AS total_invalid
FROM workspace.drs_silver.gastos_insumos_por_cultura_quarantine
""").display()
 
# COMMAND ----------
 
# Data Quality checks
dq = spark.sql("""
SELECT
    SUM(CASE WHEN culture    IS NULL THEN 1 ELSE 0 END) AS culture_nulls,
    SUM(CASE WHEN input_name IS NULL THEN 1 ELSE 0 END) AS input_name_nulls,
    SUM(CASE WHEN category   IS NULL THEN 1 ELSE 0 END) AS category_nulls,
    SUM(CASE WHEN dose_min   IS NULL
             AND dose_max    IS NULL THEN 1 ELSE 0 END) AS dose_nulls
FROM workspace.drs_silver.gastos_insumos_por_cultura
""")

display(dq)

In [0]:
# Falhar notebook se houver erro crítico de qualidade
dq_row = dq.collect()[0]
 
if (
    dq_row["culture_nulls"]    > 0 or
    dq_row["input_name_nulls"] > 0 or
    dq_row["category_nulls"]   > 0
):
    raise Exception("Data Quality check failed: workspace.drs_silver.gastos_insumos_por_cultura")
 
print("✅ Pipeline Silver concluído com sucesso!")